# Conference-Style Results Summary

This notebook summarizes `testscript.py` outputs into compact paper-ready artifacts:
1. Main summary table (overall by agent)
2. Scenario robustness heatmap (composite rank)
3. Pareto tradeoff scatter (allocation vs handovers)
4. Critical-case time series (hard scenarios only)

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


BASE_DIR = Path('.').resolve()
if not (BASE_DIR / 'BASELINE_observations_load_cycle_1.csv').exists():
    # New default location after cleanup
    if (BASE_DIR / 'test results').exists():
        BASE_DIR = BASE_DIR / 'test results'
    # Notebook may be run from repo root
    elif (BASE_DIR / 'Single Constellation ' / 'test results').exists():
        BASE_DIR = BASE_DIR / 'Single Constellation ' / 'test results'

AGENTS = ['BASELINE', 'PPO', 'DQN', 'ODT_FINETUNED']
SCENARIOS = [
    'load_cycle_1',
    'load_cycle_2',
    'load_cycle_5',
    'medium_aircraft',
    'snr_congested',
]

LATENCY_CLIP_MS = 1000  # for optional filtered latency displays

print('Base dir:', BASE_DIR)



Base dir: /Users/hindmukhtar/Documents/GitHub/DLRL2025/MultiOrbit/Test Results


In [2]:
def load_runs(base_dir: Path, agents, scenarios):
    rows = []
    missing = []

    # Ensure we point to the folder containing the CSVs
    base_dir = Path(base_dir)
    if not (base_dir / "BASELINE_observations_load_cycle_1.csv").exists():
        candidate_dirs = [
            base_dir / "test results",
            base_dir / "Single Constellation " / "test results",
            base_dir / "Single Constellation ",
        ]
        for cand in candidate_dirs:
            if (cand / "BASELINE_observations_load_cycle_1.csv").exists():
                base_dir = cand
                break

    for agent in agents:
        for scenario in scenarios:
            # Try expected file first
            candidates = [
                base_dir / f"{agent}_observations_{scenario}.csv",
            ]


            fp = next((p for p in candidates if p.exists()), None)
            if fp is None:
                missing.append(f"{agent}_observations_{scenario}.csv")
                continue

            df = pd.read_csv(fp)

            # Normalize headers from CSVs written with spaces after commas
            df.columns = (
                df.columns
                .str.strip()
                .str.replace(r"\s+", "_", regex=True)
            )
            df["agent"] = agent
            df["scenario"] = scenario

            rows.append(df)

    data = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    return data, missing, base_dir

data, missing, used_dir = load_runs(BASE_DIR, AGENTS, SCENARIOS)
print("Using dir:", used_dir)
print("Loaded rows:", len(data))
print("Loaded agent/scenario pairs:",
      data[["agent", "scenario"]].drop_duplicates().shape[0] if len(data) else 0)
if missing:
    print("Missing files (if any):", ", ".join(missing[:12]), "..." if len(missing) > 12 else "")
data.head()



Using dir: /Users/hindmukhtar/Documents/GitHub/DLRL2025/MultiOrbit/Test Results
Loaded rows: 3246
Loaded agent/scenario pairs: 3
Missing files (if any): BASELINE_observations_load_cycle_2.csv, BASELINE_observations_load_cycle_5.csv, BASELINE_observations_medium_aircraft.csv, BASELINE_observations_snr_congested.csv, PPO_observations_load_cycle_2.csv, PPO_observations_load_cycle_5.csv, PPO_observations_medium_aircraft.csv, PPO_observations_snr_congested.csv, DQN_observations_load_cycle_2.csv, DQN_observations_load_cycle_5.csv, DQN_observations_medium_aircraft.csv, DQN_observations_snr_congested.csv ...


,step,lat,lon,alt,snr,load,handovers,allocated_bw,allocation_ratio,demand_MB,...,oneweb_handover_delta,intelsat_handover_delta,oneweb_snr_db,intelsat_snr_db,oneweb_capacity_mbps,intelsat_capacity_mbps,oneweb_max_ds_mbps,intelsat_max_ds_mbps,agent,scenario
0,0,29.954182,-95.333305,365.760,18.459070,0.653756,0.0,0.000000,0.0,3.070534,...,0,0,8.761166,6.941893,150.0,50.0,150.0,50.0,BASELINE,load_cycle_1
1,1,29.951462,-95.329710,403.860,8.761977,0.632460,0.0,2.986333,1.0,2.986333,...,0,0,8.495381,7.124225,150.0,50.0,150.0,50.0,BASELINE,load_cycle_1
2,2,29.948744,-95.326120,441.960,18.936850,0.605376,1.0,2.906630,1.0,2.906630,...,1,0,8.174545,7.124902,150.0,50.0,150.0,50.0,BASELINE,load_cycle_1
3,3,29.946000,-95.322270,472.440,8.175227,0.595501,1.0,2.934174,1.0,2.934174,...,0,0,8.049398,7.151570,150.0,50.0,150.0,50.0,BASELINE,load_cycle_1
4,4,29.943250,-95.318370,501.015,20.094437,0.326068,2.0,2.865416,1.0,2.865416,...,1,0,5.809642,7.088589,150.0,50.0,150.0,50.0,BASELINE,load_cycle_1


In [3]:
data['latency_s'] = data['queing_delay_s'] + data['propagation_latency_s']

In [4]:
def episode_metrics(df):
    out = []
    for (agent, scenario), g in df.groupby(['agent', 'scenario']):
        # Active-demand slice for fair latency/allocation statistics.
        if 'demand_MB' in g.columns:
            g_active = g[g['demand_MB'] > 0].copy()
        else:
            g_active = g.copy()

        if g_active.empty:
            alloc_ratio = 1.0
            lat_violation_rate = np.nan
            avg_latency_ms = np.nan
            mean_excess_ms = np.nan
            p95_excess_ms = np.nan
            severity_score = np.nan
        else:
            alloc_ratio = float(g_active['allocation_ratio'].mean())
            # Treat sentinel queue delay (1000 s) as outage, not a valid latency sample.
            if 'queing_delay_s' in g_active.columns:
                g_active.loc[g_active['queing_delay_s'] >= 1000, 'latency_s'] = np.nan

            lat_req = g_active['latency_req_s'] if 'latency_req_s' in g_active.columns else pd.Series([np.inf] * len(g_active), index=g_active.index)
            valid_lat = g_active['latency_s'].notna()
            if valid_lat.any():
                lat_s = g_active.loc[valid_lat, 'latency_s']
                req_s = lat_req.loc[valid_lat]
                excess_s = (lat_s - req_s).clip(lower=0)
                lat_violation_rate = float((excess_s > 0).mean())
                avg_latency_ms = float(lat_s.mean() * 1000.0)
                mean_excess_ms = float(excess_s.mean() * 1000.0)
                p95_excess_ms = float(excess_s.quantile(0.95) * 1000.0)
                severity_score = float((excess_s / req_s).mean())
            else:
                lat_violation_rate = np.nan
                avg_latency_ms = np.nan
                mean_excess_ms = np.nan
                p95_excess_ms = np.nan
                severity_score = np.nan

        service_drop_s = float(g['service_drop_s'].sum()) if 'service_drop_s' in g.columns else 0.0
        total_handovers = float(g['handovers'].max()) if 'handovers' in g.columns else np.nan

        # Composite score: higher is better.
        J = (
            1.0 * alloc_ratio
            #  - 0.01 * lat_violation_rate
            # -service_drop_s
        )

        out.append({
            'agent': agent,
            'scenario': scenario,
            'allocation_ratio': alloc_ratio,
            'latency_violation_rate': lat_violation_rate,
            'service_drop_s': service_drop_s,
            'total_handovers': total_handovers,
            'avg_latency_ms': avg_latency_ms,
            'mean_excess_ms': mean_excess_ms,
            'p95_excess_ms': p95_excess_ms,
            'severity_score': severity_score,
            'composite_score': J,
        })
    return pd.DataFrame(out)

m = episode_metrics(data)
m.sort_values(['scenario','composite_score'], ascending=[True, False]).head(20)

,agent,scenario,allocation_ratio,latency_violation_rate,service_drop_s,total_handovers,avg_latency_ms,mean_excess_ms,p95_excess_ms,severity_score,composite_score
2,PPO,load_cycle_1,0.994001,0.009294,49.504470,280.0,54.644983,0.513215,0.0,inf,0.994001
0,BASELINE,load_cycle_1,0.993441,0.009294,89.854922,280.0,54.644983,0.513215,0.0,inf,0.993441
1,DQN,load_cycle_1,0.993299,0.009294,80.042828,280.0,54.644983,0.513215,0.0,inf,0.993299


## 1) Main Summary Table (Overall by Agent)

In [5]:
overall = (
    m.groupby('agent', as_index=False)
     .agg({
         'allocation_ratio': 'mean',
         'latency_violation_rate': 'mean',
         'mean_excess_ms': 'mean',
         'p95_excess_ms': 'mean',
         'severity_score': 'mean',
         'service_drop_s': 'mean',
         'total_handovers': 'mean',
         'avg_latency_ms': 'mean',
         'composite_score': 'mean',
     })
)
overall = overall.sort_values('composite_score', ascending=False)
overall.style.format({
    'allocation_ratio': '{:.4f}',
    'latency_violation_rate': '{:.2%}',
    'mean_excess_ms': '{:.2f}',
    'p95_excess_ms': '{:.2f}',
    'severity_score': '{:.4f}',
    'service_drop_s': '{:.2f}',
    'total_handovers': '{:.1f}',
    'avg_latency_ms': '{:.2f}',
    'composite_score': '{:.4f}',
})

,agent,allocation_ratio,latency_violation_rate,mean_excess_ms,p95_excess_ms,severity_score,service_drop_s,total_handovers,avg_latency_ms,composite_score
2,PPO,0.9940,0.93%,0.51,0.00,inf,49.50,280.0,54.64,0.9940
0,BASELINE,0.9934,0.93%,0.51,0.00,inf,89.85,280.0,54.64,0.9934
1,DQN,0.9933,0.93%,0.51,0.00,inf,80.04,280.0,54.64,0.9933


In [6]:
import numpy as np
import pandas as pd

df = data.copy()

# ---- Robust derived fields ----
if "allocation_ratio" not in df.columns:
    demand = pd.to_numeric(df.get("demand_MB", 0.0), errors="coerce").fillna(0.0).clip(lower=0)
    alloc = pd.to_numeric(df.get("allocated_bw", 0.0), errors="coerce").fillna(0.0).clip(lower=0)
    df["allocation_ratio"] = np.where(demand > 1e-9, alloc / demand, 1.0)
    df["allocation_ratio"] = np.clip(df["allocation_ratio"], 0.0, 1.0)

if "latency_s" not in df.columns:
    q = pd.to_numeric(df.get("queing_delay_s", 0.0), errors="coerce").fillna(0.0)
    p = pd.to_numeric(df.get("propagation_latency_s", 0.0), errors="coerce").fillna(0.0)
    df["latency_s"] = q + p

if "queing_delay_s" in df.columns:
    q = pd.to_numeric(df["queing_delay_s"], errors="coerce")
    df.loc[q >= 1000, "latency_s"] = np.nan

if "throughput_mbps" in df.columns:
    thr_col = "throughput_mbps"
elif "allocated_bw" in df.columns:
    thr_col = "allocated_bw"
elif "transmission_rate_mbps" in df.columns:
    thr_col = "transmission_rate_mbps"
else:
    raise ValueError("No throughput-like column found.")

# Active-demand steps only
active = df[pd.to_numeric(df.get("demand_MB", 0.0), errors="coerce").fillna(0.0) > 0].copy()

m = (
    active.groupby(["scenario", "agent"], as_index=False)
    .agg(
        avg_allocation_ratio=("allocation_ratio", "mean"),
        total_service_drop_s=("service_drop_s", "sum"),
        avg_throughput_mbps=(thr_col, "mean"),
        avg_latency_ms=("latency_s", lambda s: np.nanmean(s) * 1000.0),
    )
)

# Rank by allocation ratio (higher is better) within each scenario
m["rank_by_alloc"] = (
    m.groupby("scenario")["avg_allocation_ratio"]
     .rank(ascending=False, method="min")
     .astype(int)
)

# Long format -> pivot with agents as columns
long = m.melt(
    id_vars=["scenario", "agent"],
    value_vars=["rank_by_alloc", "avg_allocation_ratio", "total_service_drop_s", "avg_throughput_mbps", "avg_latency_ms"],
    var_name="metric",
    value_name="value"
)

# Optional friendly names
metric_name = {
    "rank_by_alloc": "Rank (by allocation ratio)",
    "avg_allocation_ratio": "Avg allocation/demand",
    "total_service_drop_s": "Total service drop (s)",
    "avg_throughput_mbps": "Avg throughput (Mbps)",
    "avg_latency_ms": "Avg latency (ms)",
}
long["metric"] = long["metric"].map(metric_name)

table = (
    long.pivot_table(index=["scenario", "metric"], columns="agent", values="value", aggfunc="first")
        .sort_index()
)

display(
    table.style.format({
        col: "{:.0f}" if "Rank" in str(table.index.get_level_values(1)[0]) else "{:.4f}"
        for col in table.columns
    }).format("{:.0f}", subset=pd.IndexSlice[pd.IndexSlice[:, "Rank (by allocation ratio)"], :])
      .format("{:.4f}", subset=pd.IndexSlice[pd.IndexSlice[:, "Avg allocation/demand"], :])
      .format("{:.1f}", subset=pd.IndexSlice[pd.IndexSlice[:, "Total service drop (s)"], :])
      .format("{:.2f}", subset=pd.IndexSlice[pd.IndexSlice[:, "Avg throughput (Mbps)"], :])
      .format("{:.2f}", subset=pd.IndexSlice[pd.IndexSlice[:, "Avg latency (ms)"], :])
)


## 4) Critical-Case Time Series (Hard Scenarios Only)

In [7]:
SCENARIO = "load_cycle_1"
plot_df = data[data["scenario"] == SCENARIO].copy()

if len(plot_df) == 0:
    print(f"No data found for scenario: {SCENARIO}")
else:
    plot_df = plot_df.sort_values(["agent", "step"])

    display_map = {"ODT_FINETUNED": "ODT"}

    # Plot order you requested
    plot_order = ["BASELINE", "PPO", "ODT_FINETUNED", "DQN"]

    plot_df["throughput_alloc_mbps"] = pd.to_numeric(
        plot_df.get("transmission_rate_mbps", np.nan), errors="coerce"
    )

    ref_agent = sorted(plot_df["agent"].unique())[0]
    req_df = plot_df[plot_df["agent"] == ref_agent].copy()
    req_df["throughput_req_mbps"] = pd.to_numeric(
        req_df.get("throughput_req", np.nan), errors="coerce"
    )

    plot_df["alloc_smooth"] = (
        plot_df.groupby("agent")["throughput_alloc_mbps"]
               .transform(lambda s: s.rolling(15, min_periods=1).mean())
    )
    req_df["req_smooth"] = req_df["throughput_req_mbps"].rolling(15, min_periods=1).mean()

    fig = go.Figure()

    # Add in fixed performance order
    for agent in plot_order:
        g = plot_df[plot_df["agent"] == agent]
        if g.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=g["step"],
                y=g["alloc_smooth"],
                mode="lines",
                name=display_map.get(agent, agent),
            )
        )

    # Requested plotted last (on top)
    fig.add_trace(
        go.Scatter(
            x=req_df["step"],
            y=req_df["req_smooth"],
            mode="lines",
            name="Requested Throughput",
            line=dict(color="black", dash="dash", width=3),
        )
    )

    fig.update_layout(
        height=520,
        title="Allocated vs Requested Throughput (High SNR, High Congestion)",
        xaxis_title="Step",
        yaxis_title="Throughput (Mbps)",
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.2,
            xanchor="center",
            x=0.5
        ),
        legend_title="Trace",
    )
    fig.show()


In [8]:
SCENARIO = "load_cycle_1"
plot_df = data[data["scenario"] == SCENARIO].copy()

if len(plot_df) == 0:
    print(f"No data found for scenario: {SCENARIO}")
else:
    plot_df = plot_df.sort_values(["agent", "step"])

    # Show ODT_FINETUNED as ODT in legend
    display_map = {"ODT_FINETUNED": "ODT"}

    # Requested plotting order
    plot_order = ["BASELINE", "PPO", "DQN", "ODT_FINETUNED"]

    plot_df["throughput_alloc_mbps"] = pd.to_numeric(
        plot_df.get("transmission_rate_mbps", np.nan), errors="coerce"
    )

    ref_agent = sorted(plot_df["agent"].unique())[0]
    req_df = plot_df[plot_df["agent"] == ref_agent].copy()
    req_df["throughput_req_mbps"] = pd.to_numeric(
        req_df.get("throughput_req", np.nan), errors="coerce"
    )

    plot_df["alloc_smooth"] = (
        plot_df.groupby("agent")["throughput_alloc_mbps"]
               .transform(lambda s: s.rolling(15, min_periods=1).mean())
    )
    req_df["req_smooth"] = req_df["throughput_req_mbps"].rolling(15, min_periods=1).mean()

    fig = go.Figure()

    # Add traces in fixed order
    for agent in plot_order:
        g = plot_df[plot_df["agent"] == agent]
        if g.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=g["step"],
                y=g["alloc_smooth"],
                mode="lines",
                name=display_map.get(agent, agent),
            )
        )

    # Requested Throughput last (on top)
    fig.add_trace(
        go.Scatter(
            x=req_df["step"],
            y=req_df["req_smooth"],
            mode="lines",
            name="Requested Throughput",
            line=dict(color="black", dash="dash", width=3),
        )
    )

    fig.update_layout(
        height=520,
        title="Allocated vs Requested Throughput (Large Aircraft)",
        xaxis_title="Step",
        yaxis_title="Throughput (Mbps)",
        legend_title="Trace",
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.2,
            xanchor="center",
            x=0.5
        ),
        margin=dict(b=100),
    )
    fig.show()


## 5) Per-Network Diagnostics (OneWeb vs Intelsat)
These plots use new CSV columns emitted by `testscript.py`: `oneweb_*`, `intelsat_*`.\n
If this section says columns are missing, re-run evaluations to regenerate observation CSVs.

In [9]:
required_cols = [
    'oneweb_served_mbps','intelsat_served_mbps','oneweb_share','intelsat_share',
    'oneweb_latency_s','intelsat_latency_s','oneweb_service_drop_s','intelsat_service_drop_s',
    'oneweb_handover_delta','intelsat_handover_delta','oneweb_snr_db','intelsat_snr_db',
    'oneweb_capacity_mbps','intelsat_capacity_mbps'
]
missing_cols = [c for c in required_cols if c not in data.columns]
if missing_cols:
    print('Missing per-network columns:', missing_cols)
    print('Re-run testscript.py so new columns are written to CSVs, then reload this notebook.')
else:
    print('Per-network columns available.')

# Aggregate per (scenario, agent) for paper-ready summaries
net_summary = (
    data.groupby(['scenario','agent'], as_index=False)
        .agg(
            oneweb_share=('oneweb_share','mean'),
            intelsat_share=('intelsat_share','mean'),
            oneweb_latency_ms=('oneweb_latency_s', lambda s: np.nanmean(s)*1000.0),
            intelsat_latency_ms=('intelsat_latency_s', lambda s: np.nanmean(s)*1000.0),
            oneweb_service_drop_s=('oneweb_service_drop_s','sum'),
            intelsat_service_drop_s=('intelsat_service_drop_s','sum'),
            oneweb_handover_delta=('oneweb_handover_delta','sum'),
            intelsat_handover_delta=('intelsat_handover_delta','sum')
        )
)
net_summary.head()


Per-network columns available.


,scenario,agent,oneweb_share,intelsat_share,oneweb_latency_ms,intelsat_latency_ms,oneweb_service_drop_s,intelsat_service_drop_s,oneweb_handover_delta,intelsat_handover_delta
0,load_cycle_1,BASELINE,0.511936,0.482519,46.603277,500.585700,55.341897,4.90419,273,7
1,load_cycle_1,DQN,0.472305,0.522150,46.603277,500.585700,45.529802,4.90419,273,7
2,load_cycle_1,PPO,0.407808,0.586647,41.780344,496.810277,19.895635,0.00000,273,7


In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# choose scenario
SCENARIO = "load_cycle_1"

# choose agents to plot
agents_to_plot = ["BASELINE", "PPO", "DQN", "ODT_FINETUNED"]

plot_df = data[data["scenario"] == SCENARIO].copy()
plot_df = plot_df.sort_values(["agent", "step"])

# guard
needed = ["oneweb_share", "intelsat_share", "step", "agent"]
missing = [c for c in needed if c not in plot_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Re-run testscript.py and reload data.")

# one area chart per agent
fig = make_subplots(
    rows=len(agents_to_plot), cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{a} - Network Share Over Time" for a in agents_to_plot],
    vertical_spacing=0.05,
)

for r, agent in enumerate(agents_to_plot, start=1):
    g = plot_df[plot_df["agent"] == agent]
    if g.empty:
        continue

    fig.add_trace(
        go.Scatter(
            x=g["step"], y=g["oneweb_share"],
            mode="lines", line=dict(width=0.8),
            stackgroup=f"grp{r}", name="OneWeb",
            legendgroup="OneWeb", showlegend=(r == 1)
        ),
        row=r, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=g["step"], y=g["intelsat_share"],
            mode="lines", line=dict(width=0.8),
            stackgroup=f"grp{r}", name="Intelsat",
            legendgroup="Intelsat", showlegend=(r == 1)
        ),
        row=r, col=1
    )

fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.update_layout(
    height=260 * len(agents_to_plot),
    title=f"Bandwidth Utilization Share by Network Over Time ({SCENARIO})",
    xaxis_title="Step",
    yaxis_title="Share",
)
fig.show()


In [11]:
if missing_cols:
    print('Skipping network plots until required columns are present.')
else:
    # 1) Average utilization share by source
    s1 = net_summary.melt(
        id_vars=['scenario','agent'],
        value_vars=['oneweb_share','intelsat_share'],
        var_name='network', value_name='share'
    )
    s1['network'] = s1['network'].str.replace('_share','').str.title()
    fig1 = px.bar(
        s1, x='agent', y='share', color='network', barmode='stack',
        facet_col='scenario', facet_col_wrap=3,
        title='Bandwidth Utilization Share by Network (avg over episode)'
    )
    fig1.update_yaxes(title='Share', tickformat='.0%')
    fig1.show()

    # 2) Average latency by source
    s2 = net_summary.melt(
        id_vars=['scenario','agent'],
        value_vars=['oneweb_latency_ms','intelsat_latency_ms'],
        var_name='network', value_name='latency_ms'
    )
    s2['network'] = s2['network'].str.replace('_latency_ms','').str.title()
    fig2 = px.bar(
        s2, x='agent', y='latency_ms', color='network', barmode='group',
        facet_col='scenario', facet_col_wrap=3,
        title='Per-Network Latency (ms, avg over episode)'
    )
    fig2.show()

    # 3) Service drop + handovers by source
    s3_drop = net_summary.melt(
        id_vars=['scenario','agent'],
        value_vars=['oneweb_service_drop_s','intelsat_service_drop_s'],
        var_name='network', value_name='service_drop_s'
    )
    s3_drop['network'] = s3_drop['network'].str.replace('_service_drop_s','').str.title()
    fig3 = px.bar(
        s3_drop, x='agent', y='service_drop_s', color='network', barmode='group',
        facet_col='scenario', facet_col_wrap=3,
        title='Per-Network Service Drop (s, summed over episode)'
    )
    fig3.show()

    s3_ho = net_summary.melt(
        id_vars=['scenario','agent'],
        value_vars=['oneweb_handover_delta','intelsat_handover_delta'],
        var_name='network', value_name='handover_events'
    )
    s3_ho['network'] = s3_ho['network'].str.replace('_handover_delta','').str.title()
    fig4 = px.bar(
        s3_ho, x='agent', y='handover_events', color='network', barmode='group',
        facet_col='scenario', facet_col_wrap=3,
        title='Per-Network Handovers (event count)'
    )
    fig4.show()


In [12]:
# 4) Step-level debug traces for one scenario/agent
DEBUG_SCENARIO = 'load_cycle_1'
DEBUG_AGENT = 'DQN'

if missing_cols:
    print('Skipping debug traces until required columns are present.')
else:
    dbg = data[(data['scenario'] == DEBUG_SCENARIO) & (data['agent'] == DEBUG_AGENT)].copy()
    if dbg.empty:
        print(f'No rows for {DEBUG_AGENT} / {DEBUG_SCENARIO}')
    else:
        dbg = dbg.sort_values('step')
        fig = make_subplots(rows=2, cols=2, subplot_titles=(
            'SNR by Network (dB)', 'Capacity by Network (Mbps)',
            'Served Throughput by Network (Mbps)', 'Per-step Handovers by Network'
        ))
        fig.add_trace(go.Scatter(x=dbg['step'], y=dbg['oneweb_snr_db'], name='OneWeb SNR'), row=1, col=1)
        fig.add_trace(go.Scatter(x=dbg['step'], y=dbg['intelsat_snr_db'], name='Intelsat SNR'), row=1, col=1)
        fig.add_trace(go.Scatter(x=dbg['step'], y=dbg['oneweb_capacity_mbps'], name='OneWeb Cap'), row=1, col=2)
        fig.add_trace(go.Scatter(x=dbg['step'], y=dbg['intelsat_capacity_mbps'], name='Intelsat Cap'), row=1, col=2)
        fig.add_trace(go.Scatter(x=dbg['step'], y=dbg['oneweb_served_mbps'], name='OneWeb Served'), row=2, col=1)
        fig.add_trace(go.Scatter(x=dbg['step'], y=dbg['intelsat_served_mbps'], name='Intelsat Served'), row=2, col=1)
        fig.add_trace(go.Scatter(x=dbg['step'], y=dbg['oneweb_handover_delta'], name='OneWeb HO Δ'), row=2, col=2)
        fig.add_trace(go.Scatter(x=dbg['step'], y=dbg['intelsat_handover_delta'], name='Intelsat HO Δ'), row=2, col=2)
        fig.update_layout(height=800, width=1200, title=f'Network Debug: {DEBUG_AGENT} / {DEBUG_SCENARIO}')
        fig.show()
